# Change a tuning parameter in CAM


<div class="alert alert-info">
<strong>Exercise: Modify a parameter, `micro_mg_dcs`</strong><br><br>
 

In the tuning lecture (https://www.cesm.ucar.edu/sites/default/files/2025-07/2025cesmthannay.pdf), 
we discussed the parameter **Dcs**, which controls the threshold diameter at which cloud ice particles are converted to snow. Changing this parameter affects the amount of cloud ice and therefore influences cloud radiative forcing.

For this exercise, we will use:

* **Compset:** `B1850C_LTso`
* **Resolution:** `ne16pg3_t201` (Because the resolution `ne16pg3_t201` not yet supported at this stage, use the option: `--run-unsupported`) 

To simplify the analysis, we will also interpolate all CAM output to a regular **2° × 2° latitude–longitude grid**.

Create a case called `b1850_dcs`.

Locate the namelist variable controlling **Dcs** and change it from its default value:

```fortran
micro_mg_dcs = 450.D-6
```

to

```fortran
micro_mg_dcs = 800.D-6
```

Run the model for one month.

</div>



<div class="alert alert-warning">  
<details>
   <summary>
      <font face="Times New Roman" color='blue'>Click here for hints</font>
    </summary>


### Where is the Dcs parameter?

The variable you are looking for is:

```fortran
micro_mg_dcs
```

It can be modified directly in `user_nl_cam`.

### How do I compile?

```bash
qcmd -- ./case.build
```

### Compare this run with the interpolation exercise

Compare this run with the case you created in the previous exercise:

```text
b1850_interp
```

Both cases produce output on a regular **2° × 2° latitude–longitude grid**, making them easy to compare using NetCDF tools.

You can use **ncdiff** to compute differences:

```bash
ncdiff /glade/derecho/scratch/hannay/archive/b1850_dcs/atm/hist/b1850_dcs.cam.h0a.0001-01.nc \
/glade/derecho/scratch/$USER/archive/b1850_interp/atm/hist/b1850_interp.cam.h0a.0001-01.nc \
diff.nc

```

Then visualize the result:

```bash
ncview diff.nc
```

How does changing **Dcs** affect the **LWCF** field?
</details>
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

**# create new case**
```bash
cd /glade/u/home/$USER/code/my_cesm_code/cime/scripts/

./create_newcase \
  --case ~/cases/b1850_dcs \
  --compset B1850C_LTso \
  --res ne16pg3_t201 \
  --run-unsupported
```

**# Setup**

```bash
cd ~/cases/b1850_dcs
./case.setup
```

**# Customize the CAM namelist**

Edit `user_nl_cam` and add:

```fortran
micro_mg_dcs = 800.D-6

interpolate_output = .true.
interpolate_nlat = 91
interpolate_nlon = 180
```
The first line changes the cloud microphysics parameter **Dcs**.

The remaining lines interpolate all CAM history output from the native spectral-element grid to a regular **2° × 2° latitude–longitude grid**.

**# Set the run length**

```bash
./xmlchange STOP_N=1,STOP_OPTION=nmonths
```

**# Change the job queue and project number**

If needed:

```bash
./xmlchange JOB_QUEUE=tutorial,PROJECT=UESM0015 --force
```

Use the project number provided for the tutorial.

**# Build and submit**

```bash
qcmd -- ./case.build
./case.submit
```

**# Analyze the results**

When the run completes, compare it with the `b1850_interp` case.

Create a file containing the differences:

```bash
cd /glade/derecho/scratch/$USER/archive/b1850_dcs

ncdiff \
/glade/derecho/scratch/$USER/archive/b1850_dcs/atm/hist/b1850_dcs.cam.h0.0001-01.nc \
/glade/derecho/scratch/$USER/archive/b1850_interp/atm/hist/b1850_interp.cam.h0.0001-01.nc \
diff.nc
```

Visualize the differences:

```bash
ncview diff.nc
```

Display the **LWCF** field and examine how it changes.

**# Discussion**

**Dcs** is the threshold diameter at which cloud ice particles are converted to snow.

Increasing

```fortran
micro_mg_dcs = 450.D-6
```

to

```fortran
micro_mg_dcs = 800.D-6
```

allows cloud ice particles to grow larger before being converted to snow. As a result, more cloud ice tends to remain suspended in the atmosphere, which generally increases the **Longwave Cloud Forcing (LWCF)**.

The largest changes are typically found in the tropical deep convective regions, where high ice clouds are abundant.

Keep in mind that this simulation is only one month long. Although you should see differences in the LWCF field, a one-month simulation is too short to draw statistically robust conclusions about the climate response.


**#How does this affect the LWCF ?** 

Dcs = Threshold diameter to convert cloud ice particles to snow

*<p style="text-align: center;"> 
    ![dcs representation](../../../images/namelist/dcs.png)  
    Figure: Representation of Dcs. </p>*
    
We increased Dcs from 450 microns to 800 microns. We should have more ice cloud and the LWCF should be larger. 

  
The field `LWCF` difference looks like 
*<p style="text-align: center;"> 
    ![ncview diff LWCF](../../../images/namelist/ncview_diff_dcs.png)  
    Figure: Difference of LWCF for micro_mg_dcs = 450.D-6->800.D-6 </p>*
    
We can see that the change in Dcs affects the tropics where LWCF is large. However, 1-month run is too short to look at robust statistics. 

</details>
</div>

